In [1]:
import mlflow
import os
import pickle
from mlflow.tracking import MlflowClient
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error


In [2]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")

client = MlflowClient(tracking_uri='http://127.0.0.1:5000')

def load_pickle(filename: str):
    with open(filename, 'rb') as f_in:
        return pickle.load(f_in)

In [10]:
def train_log_model(data_path: str, params):
    X_train, y_train = load_pickle(os.path.join(data_path, 'train.pkl'))
    X_val, y_val = load_pickle(os.path.join(data_path, 'val.pkl'))
    X_test, y_test = load_pickle(os.path.join(data_path, 'test.pkl'))


    with mlflow.start_run():
        #Log parameters
        mlflow.log_params(params)


        rf = RandomForestRegressor(**params)
        rf.fit(X_train, y_train)

        #Evaluate on validation set
        val_rmse = root_mean_squared_error(y_val, rf.predict(X_val))
        mlflow.log_metric('val_rmse', val_rmse)

        #Evaluate on test set
        test_rmse = root_mean_squared_error(y_test, rf.predict(X_test))
        mlflow.log_metric('test_rmse', test_rmse)

        #Log model
        mlflow.sklearn.log_model(rf, artifact_path='models')

        return mlflow.active_run().info.run_id


In [11]:
def register_model(data_path: str, top_n: int = 5):
    
    #Get experiment from hyperparameter optimization
    experiment = client.get_experiment_by_name('homework-random-forest-hyperopt')

    #Get top n runs sorted by validation RMSE
    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=['metrics.val_rmse ASC'],
        max_results=top_n
    )

    print(f"Found {len(runs)} top runs from hyperparameter optimization experiment.")

    #create new experiment for registered models
    mlflow.set_experiment('homework-registered-models')

    #Train and log models for top n runs

    for run in runs:
        params = {
            'max_depth': int(run.data.params['max_depth']),
            'n_estimators': int(run.data.params['n_estimators']),
            'min_samples_split': int(run.data.params['min_samples_split']),
            'min_samples_leaf': int(run.data.params['min_samples_leaf']),
            'random_state': 42
        }
        train_log_model(data_path, params)

    #select the best model (lowest RMSE)

    best_experiment = client.get_experiment_by_name('homework-registered-models')

    best_run = client.search_runs(
        experiment_ids=[best_experiment.experiment_id],
        order_by=['metrics.test_rmse ASC'],
        max_results=1
    )[0]

    best_test_rmse = best_run.data.metrics['test_rmse']
    best_run_id = best_run.info.run_id

    print(f"Best run ID: {best_run_id} with test RMSE: {best_test_rmse:.4f}")


    #Register the best model
    model_uri = f"runs:/{best_run_id}/models"
    mlflow.register_model(model_uri=model_uri, name='homework-best-random-forest')

    return best_test_rmse

data_path = '../output'
best_rmse = register_model(data_path, top_n=5)

Found 5 top runs from hyperparameter optimization experiment.


2026/03/28 15:59:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 15:59:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run adaptable-kite-921 at: http://127.0.0.1:5000/#/experiments/3/runs/c11649aca89b451e8f89b42315976dfb
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/03/28 15:59:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 15:59:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run bright-steed-871 at: http://127.0.0.1:5000/#/experiments/3/runs/64bc1b640c794754b92d84ea4480de58
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


...

2026/03/28 16:00:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 16:00:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run skillful-turtle-425 at: http://127.0.0.1:5000/#/experiments/3/runs/db860d5546a74bba98b753c161c4f914
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/03/28 16:00:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 16:00:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run bold-crow-335 at: http://127.0.0.1:5000/#/experiments/3/runs/8232f4234239415eb93d98854d7b05db
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
Best run ID: 8232f4234239415eb93d98854d7b05db with test RMSE: 5.5783


Successfully registered model 'homework-best-random-forest'.
2026/03/28 16:00:21 WARNING mlflow.tracking._model_registry.fluent: Run with id 8232f4234239415eb93d98854d7b05db has no artifacts at artifact path 'models', registering model based on models:/m-83bfff6e9b7d4beba0d704504c75bf52 instead
2026/03/28 16:00:21 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: homework-best-random-forest, version 1
Created version '1' of model 'homework-best-random-forest'.
